# IADD Traffic Detection - YOLOv11 (main pipeline)

One notebook, run top to bottom. Set RUN_NAME / model / hyper-parameters in the config
cell, then:
1. unzip the dataset and REMOVE the leaked near-duplicate frames (eval_exclude.txt),
2. train (with a low patience so it early-stops soon after it converges),
3. use the convergence-check cell to know when to stop the training cell,
4. run the finish fine-tune (mosaic/mixup off) for the final boost,
5. leakage-free evaluation: honest mAP + confusion matrix at the F1-optimal confidence,
   for val, test, and the pooled val+test set.

Upload `eval_exclude.txt` to the root of your Drive before running.

## 1. GPU / install / mount

In [ ]:
!nvidia-smi
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

## 2. Config - edit this per run

In [ ]:
MODEL    = 'yolo11s.pt'   # yolo11s / yolo11m ...
IMGSZ    = 1280
BATCH    = 12
EPOCHS   = 150
PATIENCE = 10             # was 20; 10 stops ~5-6 h after convergence instead of ~11 h

RUN_NAME = 'run7'         # <-- change for each new run
DRIVE    = '/content/drive/MyDrive'
RUNS     = f'{DRIVE}/iadd_runs'
DS       = '/content/iadd_subset_v4'
DATA     = f'{DS}/data.yaml'

## 3. Unzip the dataset (re-run after any reconnect)

In [ ]:
!rm -rf /content/iadd_subset_v4
!unzip -q "/content/drive/MyDrive/iadd_subset_v4.zip" -d /content

## 4. Remove leaked near-duplicate frames

IADD has a few duplicate videos under different Record IDs, so some val/test frames have
a near-identical twin in train. We delete those frames (found by perceptual hashing,
listed in eval_exclude.txt) so BOTH training-time val and the final eval are clean.

In [ ]:
import os, glob
lines = [l.strip() for l in open(f'{DRIVE}/eval_exclude.txt') if l.strip()]
removed = 0
for entry in lines:
    split, stem = entry.split('/', 1)
    for sub, ext in (('images', '.jpg'), ('labels', '.txt')):
        p = f'{DS}/{sub}/{split}/{stem}{ext}'
        if os.path.exists(p):
            os.remove(p); removed += 1
print('removed', removed, 'files from', len(lines), 'leaked frames')
for s in ('val', 'test'):
    print(f'  clean {s}: {len(glob.glob(f"{DS}/images/{s}/*.jpg"))} images')
# clear any stale label caches so the removed frames are not read back
for c in glob.glob(f'{DS}/labels/*.cache'):
    os.remove(c)

## 5. Train - fresh, or auto-resume from Drive

In [ ]:
from ultralytics import YOLO
import os

CKPT = f'{RUNS}/{RUN_NAME}/weights/last.pt'
if os.path.exists(CKPT):
    print('checkpoint found -> resuming:', CKPT)
    model = YOLO(CKPT); model.train(resume=True)
else:
    print('no checkpoint -> fresh run:', MODEL, '| imgsz', IMGSZ, '| batch', BATCH)
    model = YOLO(MODEL)
    model.train(
        data=DATA, epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
        project=RUNS, name=RUN_NAME, exist_ok=True, plots=True,
        mixup=0.1, hsv_v=0.4, degrees=5.0,   # calibrated augmentation
    )

## 6. Convergence check - run this any time to decide when to stop

A cell cannot interrupt the blocking train cell, and frequent multi-account resumes can
reset the early-stopping counter, so use this to decide when to stop training manually.

In [ ]:
import pandas as pd
df = pd.read_csv(f'{RUNS}/{RUN_NAME}/results.csv'); df.columns = [c.strip() for c in df.columns]
fit = 0.1 * df['metrics/mAP50(B)'] + 0.9 * df['metrics/mAP50-95(B)']
best_i = int(fit.idxmax()); since = len(fit) - 1 - best_i
print(f'epochs done: {len(df)}   best at epoch {int(df["epoch"].iloc[best_i])}   ({since} epochs ago)')
print('mAP50    last 6:', [round(x, 4) for x in df['metrics/mAP50(B)'].tail(6)])
print('mAP50-95 last 6:', [round(x, 4) for x in df['metrics/mAP50-95(B)'].tail(6)])
print('\n>>> CONVERGED - safe to STOP the train cell <<<' if since >= 8
      else '>>> still improving - keep going <<<')

## 7. Finish fine-tune - the final jump (run AFTER convergence)

Continues from the run's best.pt for ~10 epochs with mosaic + mixup OFF and a low
learning rate (set explicitly so it is not overridden by optimizer='auto'). This is the
close_mosaic effect that the resumes prevented; it polishes on realistic single images.
Output = RUN_NAME + '_ft'.

In [ ]:
from ultralytics import YOLO
import os

FT_CKPT = f'{RUNS}/{RUN_NAME}_ft/weights/last.pt'
if os.path.exists(FT_CKPT):
    print('resuming finish ->', FT_CKPT)
    YOLO(FT_CKPT).train(resume=True)
else:
    print('finish fine-tune from', f'{RUN_NAME}/weights/best.pt')
    YOLO(f'{RUNS}/{RUN_NAME}/weights/best.pt').train(
        data=DATA, epochs=10, imgsz=IMGSZ, batch=BATCH,
        project=RUNS, name=f'{RUN_NAME}_ft', exist_ok=True, plots=True,
        optimizer='SGD', lr0=0.001, lrf=0.01, warmup_epochs=0.0,
        mosaic=0.0, mixup=0.0, hsv_v=0.4, degrees=5.0,
    )

## 8. Leakage-free evaluation - val, test, pooled

Uses the finished model if it exists, else the base run. Honest mAP at the default
threshold, the confusion matrix at the F1-optimal confidence (read from the val curve),
and the pooled val+test number as the robust headline.

In [ ]:
from ultralytics import YOLO
import numpy as np, os, shutil, glob, yaml

FT = f'{RUNS}/{RUN_NAME}_ft/weights/best.pt'
best = YOLO(FT if os.path.exists(FT) else f'{RUNS}/{RUN_NAME}/weights/best.pt')
TAG = f'{RUN_NAME}_eval'

# honest mAP on clean val AND test first (creates both folders)
mv = best.val(data=DATA, split='val',  plots=True, project=RUNS, name=f'{TAG}_val',  exist_ok=True)
print('VAL  mAP50=%.4f  mAP50-95=%.4f' % (mv.box.map50, mv.box.map))
mt = best.val(data=DATA, split='test', plots=True, project=RUNS, name=f'{TAG}_test', exist_ok=True)
print('TEST mAP50=%.4f  mAP50-95=%.4f' % (mt.box.map50, mt.box.map))

# operating confidence from val F1 curve, then overwrite the confusion matrix at it
CONF = 0.25
for x, y, _, yl in mv.curves_results:
    if yl == 'F1':
        px = np.asarray(x); y = np.asarray(y); f1 = y.mean(0) if y.ndim > 1 else y
        CONF = round(float(px[f1.argmax()]), 3)
print('operating confidence (val F1) =', CONF)
for split in ('val', 'test'):
    best.val(data=DATA, split=split, conf=CONF, plots=True, project=RUNS, name=f'{TAG}_{split}_tmp', exist_ok=True)
    for f in ('confusion_matrix.png', 'confusion_matrix_normalized.png'):
        shutil.copy(f'{RUNS}/{TAG}_{split}_tmp/{f}', f'{RUNS}/{TAG}_{split}/{f}')
    shutil.rmtree(f'{RUNS}/{TAG}_{split}_tmp')

# pooled val+test = robust headline
POOL = '/content/iadd_pool'
os.system(f'rm -rf {POOL}'); os.makedirs(f'{POOL}/images/pool'); os.makedirs(f'{POOL}/labels/pool')
for sp in ('val', 'test'):
    for img in glob.glob(f'{DS}/images/{sp}/*.jpg'):
        b = os.path.basename(img); os.symlink(img, f'{POOL}/images/pool/{b}')
        lb = f'{DS}/labels/{sp}/' + b[:-4] + '.txt'
        if os.path.exists(lb): os.symlink(lb, f'{POOL}/labels/pool/' + b[:-4] + '.txt')
yaml.safe_dump({'path': POOL, 'train': 'images/pool', 'val': 'images/pool', 'nc': 6,
                'names': ['person','car','motorcycle','bus','truck','traffic_light']}, open(f'{POOL}/data.yaml', 'w'))
mp = best.val(data=f'{POOL}/data.yaml', split='val', plots=False, project=RUNS, name=f'{TAG}_pool', exist_ok=True)
print('\nPOOLED (val+test)  mAP50=%.4f  mAP50-95=%.4f' % (mp.box.map50, mp.box.map))